**Import tableau final**

In [ ]:
!pip install --upgrade google-cloud-bigquery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project="projet-riskhorizon-2276")


In [52]:
query = """
SELECT *
FROM `projet-riskhorizon-2276.3_Mart.tableau_final_table`
"""

df = client.query(query).to_dataframe()

df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,AGE,YEARS_EMPLOYED,tx_endettement,Reste_a_vivre,Reste_a_vivre_par_personne,AVG_CREDIT_SCORE,APPORT_ESTIME,RATIO_PRET,nb_credits_ext,nb_credits_actifs_ext,nb_credits_clotures_ext,total_credit_ext,dette_totale_ext,ratio_dette_credit_ext,nb_credits_en_retard_ext,retard_max_ext,montant_total_retard_ext,nb_prolongations_ext,nb_types_credit_ext,nb_demandes_passees_int,nb_refus_passes_int,montant_total_emprunte_passe_int
0,355557,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4450.5,45000.0,PENSIONER,SECONDARY_/_SECONDARY_SPECIAL,MARRIED,UNKNOWN,NaN,0.060382,NaN,55.0,0.0,16.48,1879.125,939.562,0.08,0.0,1.0,3,2,1,257368.77,122050.98,0.474226,0,0,0.0,0,2,1,0,75312.0
1,211865,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4383.0,45000.0,PENSIONER,SECONDARY_/_SECONDARY_SPECIAL,MARRIED,UNKNOWN,0.734834,0.716274,0.614414,55.6,0.0,16.23,1884.750,942.375,0.08,0.0,1.0,17,6,10,1512477.00,637623.00,0.421575,0,0,0.0,0,2,6,0,660501.0
2,364118,0,CASH_LOANS,F,False,True,0,29250.0,45000.0,4855.5,45000.0,PENSIONER,SECONDARY_/_SECONDARY_SPECIAL,MARRIED,UNKNOWN,NaN,0.298588,NaN,61.3,0.0,16.60,2032.875,1016.438,0.08,0.0,1.0,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,5,2,207724.5
3,143626,1,CASH_LOANS,F,False,True,0,31500.0,45000.0,5076.0,45000.0,STATE_SERVANT,SECONDARY_/_SECONDARY_SPECIAL,MARRIED,MEDICINE,NaN,0.373220,NaN,22.3,2.8,16.11,2202.000,1101.000,0.08,0.0,1.0,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,1,0,32503.5
4,372569,0,CASH_LOANS,F,False,False,0,31500.0,45000.0,4450.5,45000.0,PENSIONER,HIGHER_EDUCATION,WIDOW,UNKNOWN,0.888203,0.684629,0.758393,65.0,0.0,14.13,2254.125,2254.125,0.08,0.0,1.0,17,7,10,633280.14,29817.00,0.047083,0,0,0.0,0,2,2,0,207616.5


In [54]:
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score


# --- 1. PRÉPARATION DES DONNÉES --
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

for col in categorical_features:
    X[col] = X[col].astype('category')

# Sauvegarde des noms de variables pour l'interprétation
feature_names = X.columns

# Split initial
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#---CRÉATION DU PIPELINE --- class_weight="balanced",

model_pipeline = Pipeline([
    ('classifier', HistGradientBoostingClassifier(
        max_iter=200,
        max_depth=5,
        min_samples_leaf=50,
        class_weight="balanced",
        categorical_features='from_dtype', # <-- CRUCIAL : dit au modèle de détecter automatiquement les colonnes 'category'
        random_state=42
    ))
])

# --- 3. VALIDATION CROISÉE ---

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print("VALIDATION CROISÉE (Gestion Native du Catégoriel)")
print(f"Scores AUC par fold : {cv_scores}")
print(f"Score AUC Moyen : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print("-" * 30)

# --- 4. ENTRAÎNEMENT FINAL & ÉVALUATION ---

model_pipeline.fit(X_train, y_train)

y_pred_test = model_pipeline.predict_proba(X_test)[:, 1]
print(f"AUC Test Final : {roc_auc_score(y_test, y_pred_test):.4f}\n")


VALIDATION CROISÉE (Gestion Native du Catégoriel)
Scores AUC par fold : [0.76248371 0.76081481 0.76369738 0.75942315 0.76073627]
Score AUC Moyen : 0.7614 (+/- 0.0030)
------------------------------
AUC Test Final : 0.7611



**AUC** La probabilité du bon choix ($76,11\%$) : Si on tire au sort un "vrai" mauvais payeur d'un côté, et un "vrai" bon payeur de l'autre, le modèle attribuera un score de risque plus élevé au mauvais payeur dans $76,11\%$ des cas.

In [56]:
# --- 5. APPLICATION DU SEUIL BANCAIRE ---
SEUIL_BANQUE = 0.5  # Ajustable selon vos besoins métiers
y_pred_test_ajuste = (y_pred_test >= SEUIL_BANQUE).astype(int)

print(f"--- PERFORMANCES GLOBALES (SEUIL IMPOSE : {SEUIL_BANQUE}) ---")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_test_ajuste):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_test_ajuste):.4f}")
print(f"Précision : {precision_score(y_test, y_pred_test_ajuste):.4f}\n")

# Masques pour l'équité des genres
masque_hommes = (df.loc[X_test.index, "CODE_GENDER"] == "M")
masque_femmes = (df.loc[X_test.index, "CODE_GENDER"] == "F")

def afficher_scores_par_genre(genre, masque):
    print(f"--- PERFORMANCES : {genre} ---")
    print(f"Nombre d'individus : {sum(masque)}")
    print(f"Recall    : {recall_score(y_test[masque], y_pred_test_ajuste[masque]):.4f}")
    print(f"Précision : {precision_score(y_test[masque], y_pred_test_ajuste[masque]):.4f}\n")

afficher_scores_par_genre("HOMMES (M)", masque_hommes)
afficher_scores_par_genre("FEMMES (F)", masque_femmes)

--- PERFORMANCES GLOBALES (SEUIL IMPOSE : 0.5) ---
Accuracy  : 0.7191
Recall    : 0.6681
Précision : 0.1751

--- PERFORMANCES : HOMMES (M) ---
Nombre d'individus : 20931
Recall    : 0.7581
Précision : 0.1856

--- PERFORMANCES : FEMMES (F) ---
Nombre d'individus : 40569
Recall    : 0.6011
Précision : 0.1662



Proba par clients et selections des variables les plus influentes

In [57]:
# --- 6. INTERPRÉTABILITÉ SHAP ---
# Préparation d'une copie numérique stricte pour SHAP
X_full_df = X.copy()
for col in X_full_df.columns:
    if X_full_df[col].dtype.name == 'category':
        X_full_df[col] = X_full_df[col].cat.codes.astype(float)
    else:
        X_full_df[col] = X_full_df[col].astype(float)

# CORRECTION ICI : Extraction directe du classifieur depuis le pipeline
final_classifier = model_pipeline.named_steps['classifier']

# Calcul des valeurs SHAP
explainer = shap.TreeExplainer(final_classifier)
shap_values_full = explainer(X_full_df)
shap_full_df = pd.DataFrame(shap_values_full.values, columns=X_full_df.columns, index=df.index)

# Enregistrement des probabilités et décisions sur le df principal
df['proba_defaut'] = model_pipeline.predict_proba(X)[:, 1]
df['PREDICTION_FINALE'] = (df['proba_defaut'] >= SEUIL_BANQUE).astype(int)

# Extraction du Top 3 SHAP par individu
shap_abs = shap_full_df.abs()
top_3_indices = np.argsort(shap_abs.values, axis=1)[:, ::-1][:, :3]

for i in range(3):
    rang = i + 1
    df[f'VAR_DOMINANTE_{rang}'] = shap_full_df.columns[top_3_indices[:, i]]
    df[f'IMPACT_COEF_{rang}'] = shap_full_df.values[np.arange(len(df)), top_3_indices[:, i]]
    df[f'SENS_IMPACT_{rang}'] = np.where(df[f'IMPACT_COEF_{rang}'] > 0, "Aggrave le risque", "Sécurise le dossier")

In [59]:
# --- 7. EXTRACTION DU DATAFRAME FINAL RESTREINT ---
colonnes_var_dominante = ['VAR_DOMINANTE_1', 'VAR_DOMINANTE_2', 'VAR_DOMINANTE_3']
colonnes_uniques = df[colonnes_var_dominante].stack().unique().tolist()

# Construction dynamique de la liste des colonnes cibles
toutes_les_colonnes = ["SK_ID_CURR", "TARGET", "proba_defaut", "PREDICTION_FINALE"]

for rang in range(1, 4):
    toutes_les_colonnes.extend([f'VAR_DOMINANTE_{rang}', f'IMPACT_COEF_{rang}', f'SENS_IMPACT_{rang}'])

toutes_les_colonnes.extend(list(colonnes_uniques))
toutes_les_colonnes = list(dict.fromkeys(toutes_les_colonnes)) # Élimine les doublons de liste

# Création et affichage du DataFrame d'export
df_final = df[toutes_les_colonnes].copy()

print("=== INSPECTION DU DATAFRAME DE SORTIE ===")
df_final.head()

=== INSPECTION DU DATAFRAME DE SORTIE ===


,SK_ID_CURR,TARGET,proba_defaut,PREDICTION_FINALE,VAR_DOMINANTE_1,IMPACT_COEF_1,SENS_IMPACT_1,VAR_DOMINANTE_2,IMPACT_COEF_2,SENS_IMPACT_2,VAR_DOMINANTE_3,IMPACT_COEF_3,SENS_IMPACT_3,EXT_SOURCE_2,NAME_EDUCATION_TYPE,ORGANIZATION_TYPE,EXT_SOURCE_1,FLAG_OWN_REALTY,EXT_SOURCE_3,AMT_CREDIT,total_credit_ext,AMT_INCOME_TOTAL,dette_totale_ext,nb_types_credit_ext,CODE_GENDER,AGE,nb_refus_passes_int,ratio_dette_credit_ext,YEARS_EMPLOYED,tx_endettement,montant_total_emprunte_passe_int,RATIO_PRET,NAME_FAMILY_STATUS,nb_credits_clotures_ext,nb_demandes_passees_int,montant_total_retard_ext,nb_credits_actifs_ext,CNT_CHILDREN,NAME_INCOME_TYPE,NAME_CONTRACT_TYPE,nb_prolongations_ext,AMT_GOODS_PRICE,Reste_a_vivre,nb_credits_ext,FLAG_OWN_CAR,APPORT_ESTIME
0,355557,0,0.456912,0,EXT_SOURCE_2,1.148086,Aggrave le risque,NAME_EDUCATION_TYPE,-0.233384,Sécurise le dossier,ORGANIZATION_TYPE,-0.219850,Sécurise le dossier,0.060382,SECONDARY_/_SECONDARY_SPECIAL,UNKNOWN,NaN,True,NaN,45000.0,257368.77,27000.0,122050.98,2,F,55.0,0,0.474226,0.0,16.48,75312.0,1.0,MARRIED,1,1,0.0,2,0,PENSIONER,CASH_LOANS,0,45000.0,1879.125,3,False,0.0
1,211865,0,0.079296,0,EXT_SOURCE_1,-0.505995,Sécurise le dossier,EXT_SOURCE_2,-0.385956,Sécurise le dossier,ORGANIZATION_TYPE,-0.330033,Sécurise le dossier,0.716274,SECONDARY_/_SECONDARY_SPECIAL,UNKNOWN,0.734834,True,0.614414,45000.0,1512477.00,27000.0,637623.00,2,F,55.6,0,0.421575,0.0,16.23,660501.0,1.0,MARRIED,10,6,0.0,6,0,PENSIONER,CASH_LOANS,0,45000.0,1884.750,17,False,0.0
2,364118,0,0.349050,0,EXT_SOURCE_2,0.486188,Aggrave le risque,ORGANIZATION_TYPE,-0.278948,Sécurise le dossier,NAME_EDUCATION_TYPE,-0.232392,Sécurise le dossier,0.298588,SECONDARY_/_SECONDARY_SPECIAL,UNKNOWN,NaN,True,NaN,45000.0,NaN,29250.0,NaN,<NA>,F,61.3,2,NaN,0.0,16.60,207724.5,1.0,MARRIED,<NA>,5,NaN,<NA>,0,PENSIONER,CASH_LOANS,<NA>,45000.0,2032.875,<NA>,False,0.0
3,143626,1,0.451930,0,EXT_SOURCE_2,0.357108,Aggrave le risque,NAME_EDUCATION_TYPE,-0.283557,Sécurise le dossier,FLAG_OWN_REALTY,-0.246046,Sécurise le dossier,0.373220,SECONDARY_/_SECONDARY_SPECIAL,MEDICINE,NaN,True,NaN,45000.0,NaN,31500.0,NaN,<NA>,F,22.3,0,NaN,2.8,16.11,32503.5,1.0,MARRIED,<NA>,1,NaN,<NA>,0,STATE_SERVANT,CASH_LOANS,<NA>,45000.0,2202.000,<NA>,False,0.0
4,372569,0,0.106769,0,EXT_SOURCE_1,-0.853094,Sécurise le dossier,FLAG_OWN_REALTY,-0.762770,Sécurise le dossier,EXT_SOURCE_3,-0.475355,Sécurise le dossier,0.684629,HIGHER_EDUCATION,UNKNOWN,0.888203,False,0.758393,45000.0,633280.14,31500.0,29817.00,2,F,65.0,0,0.047083,0.0,14.13,207616.5,1.0,WIDOW,10,2,0.0,7,0,PENSIONER,CASH_LOANS,0,45000.0,2254.125,17,False,0.0


In [60]:
# 1. On récupère la liste de TOUTES les colonnes du DataFrame d'origine (df)
toutes_les_colonnes_origine = df.columns.tolist()

# 2. On trouve les colonnes exclues (celles qui sont dans l'origine mais PAS dans df_final)
colonnes_non_selectionnees = [col for col in toutes_les_colonnes_origine if col not in df_final.columns]

# --- AFFICHAGE DU BILAN ---
print("=== ANALYSE DES COLONNES SUPPRIMÉES ===")
print(f"Nombre de colonnes supprimées : {len(colonnes_non_selectionnees)}")
print("-" * 40)

# Affichage de la liste des colonnes jetées (par blocs de 5 pour que ce soit lisible)
for i in range(0, len(colonnes_non_selectionnees), 5):
    print(colonnes_non_selectionnees[i:i+5])

print("-" * 40)

# 3. CRÉATION DU DATAFRAME DES COLONNES EXCLUES
# On conserve souvent 'SK_ID_CURR' comme clé de jointure au cas où
colonnes_a_garder_exclues = ["SK_ID_CURR"] + colonnes_non_selectionnees
df_exclues = df[colonnes_a_garder_exclues].copy()

print(f"Dimensions du DataFrame des colonnes exclues (df_exclues) : {df_exclues.shape}")

=== ANALYSE DES COLONNES SUPPRIMÉES ===
Nombre de colonnes supprimées : 5
----------------------------------------
['AMT_ANNUITY', 'Reste_a_vivre_par_personne', 'AVG_CREDIT_SCORE', 'nb_credits_en_retard_ext', 'retard_max_ext']
----------------------------------------
Dimensions du DataFrame des colonnes exclues (df_exclues) : (307497, 6)


In [61]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "tableau_final_HistGradientBoostingClassifier_v1"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQueryS
df_final.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_4706/1829066182.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 9776.93it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.tableau_final_HistGradientBoostingClassifier_v1


**Etape 10 : Matrice de confusion**

In [44]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# conversion des probabilités en classes (Seuil par défaut à 0.5)
y_pred_class = (y_pred_test >= 0.08).astype(int)

# calcul de la matrice brute et normalisée
cm = confusion_matrix(y_test, y_pred_class)
cm_normalized = confusion_matrix(y_test, y_pred_class, normalize='true') # Taux par ligne

# création du graphique Plotly
labels_etats = ['Non-Défaut (0)', 'Défaut (1)']

fig_cm = px.imshow(
    cm_normalized,
    text_auto=".2%", # Affiche les taux en pourcentage
    x=labels_etats,
    y=labels_etats,
    labels=dict(x="État PRÉDIT par le modèle", y="État RÉEL du client", color="Taux"),
    title="Matrice de Confusion (Transition Réel vs Prédit)",
    color_continuous_scale="Blues"
)

fig_cm.show()

***Nouvelle table avec coefficient par client :***

In [ ]:
# 1. Calcule directement les probabilités sur X_encoded (le pipeline gère le scaling en interne)
proba_full = model_pipeline.predict_proba(X_encoded)[:, 1]

# 2. On calcule le score de confiance (plus il est proche de 1, plus le client est sûr)
score_full = 1 - proba_full

# 3. Création de ton DataFrame final avec les résultats
df_final = df.copy()
df_final["proba_defaut"] = proba_full
df_final["score"] = score_full

In [ ]:
df_final.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,retard_max_ext,montant_total_retard_ext,nb_prolongations_ext,nb_types_credit_ext,nb_demandes_passees_int,nb_refus_passes_int,montant_total_emprunte_passe_int,EDUCATION_RANK,proba_defaut,score
0,355557,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4450.5,...,0,0.0,0,2,1,0,75312.0,1.0,0.599416,0.400584
1,211865,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4383.0,...,0,0.0,0,2,6,0,660501.0,1.0,0.131708,0.868292
2,364118,0,CASH_LOANS,F,False,True,0,29250.0,45000.0,4855.5,...,<NA>,NaN,<NA>,<NA>,5,2,207724.5,1.0,0.440325,0.559675
3,143626,1,CASH_LOANS,F,False,True,0,31500.0,45000.0,5076.0,...,<NA>,NaN,<NA>,<NA>,1,0,32503.5,1.0,0.446742,0.553258
4,372569,0,CASH_LOANS,F,False,False,0,31500.0,45000.0,4450.5,...,0,0.0,0,2,2,0,207616.5,3.0,0.077283,0.922717


In [ ]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "tableau_final_LogisticRegression_v3"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQueryS
df_final.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_422/682671322.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 8612.53it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.tableau_final_LogisticRegression_v3


In [ ]:
# from sklearn.model_selection import GridSearchCV

# # --- 3. RECHERCHE AUTOMATIQUE DES MEILLEURS PARAMÈTRES (GRID SEARCH) ---

# # On définit la grille des paramètres à tester
# # Notez le préfixe 'classifier__' obligatoire ici
# param_grid = {
#     'classifier__max_iter': [100, 150, 200],
#     'classifier__max_depth': [3, 5, 8, None],
#     'classifier__min_samples_leaf': [20, 50, 100]
# }
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# # On configure le GridSearch avec votre pipeline et la métrique ROC AUC
# grid_search = GridSearchCV(
#     estimator=model_pipeline,
#     param_grid=param_grid,
#     cv=cv,                 # Utilise votre StratifiedKFold déjà défini
#     scoring='roc_auc',     # Votre métrique de choix !
#     n_jobs=-1,             # Utilise tous les cœurs de votre processeur pour aller plus vite
#     verbose=1              # Affiche l'avancement des calculs
# )

# print("Lancement de la recherche des meilleurs paramètres...")
# grid_search.fit(X_train, y_train)

# # --- 4. RÉSULTATS & ÉVALUATION FINALE ---

# print("\n--- RÉSULTATS DE LA RECHERCHE ---")
# print(f"Meilleur score AUC en validation : {grid_search.best_score_:.4f}")
# print(f"Meilleurs paramètres trouvés : {grid_search.best_params_}")
# print("-" * 30)

# # Le 'grid_search' contient automatiquement le meilleur modèle entraîné sur tout le bloc Train.
# # On l'utilise directement pour prédire sur le jeu de Test final.
# best_model = grid_search.best_estimator_
# y_pred_test = best_model.predict_proba(X_test)[:, 1]

# print(f"AUC Test Final (avec les meilleurs paramètres) : {roc_auc_score(y_test, y_pred_test):.4f}")